# 09 - Secondary Outcome: Opioid Overdose Deaths

Does cannabis legalization affect opioid overdose mortality? The 'substitution hypothesis' predicts legal cannabis reduces opioid demand. Run the same DiD designs on CDC WONDER overdose data.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from linearmodels.panel import PanelOLS

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

CDC_FILE = "cdc_state_year.parquet"
if not (DATA_DIR / CDC_FILE).exists():
    raise FileNotFoundError(
        "CDC panel not found. Run:\n"
        "  python scripts/download_cdc.py  (follow manual instructions)\n"
        "  python src/build_cdc_panel.py"
    )
cdc = pd.read_parquet(DATA_DIR / CDC_FILE)
leg = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"CDC panel: {cdc.shape}")

## Overdose trends: treated vs never-treated states

In [ ]:
outcome = "overdose_deaths_per_100k" if "overdose_deaths_per_100k" in cdc.columns else "overdose_deaths"

treated_states = leg[leg['retail_sales_year'].notna()]['state'].tolist()
avg = (
    cdc
    .assign(group=cdc['state'].map(lambda s: 'Treated' if s in treated_states else 'Never Treated'))
    .groupby(['year','group'])[outcome]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11,5))
for grp, color in [('Treated','steelblue'),('Never Treated','orange')]:
    sub = avg[avg['group']==grp]
    ax.plot(sub['year'], sub[outcome], marker='o', ms=4, lw=1.8, label=grp, c=color)
ax.set_xlabel("Year")
ax.set_ylabel(outcome.replace('_',' ').title())
ax.set_title("Opioid Overdose Trends: Treated vs Never-Treated States")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "09_overdose_trends.png", bbox_inches='tight')
plt.show()

## TWFE DiD on overdose deaths

In [ ]:
never = leg[leg['retail_sales_year'].isna()]['state'].tolist()
in_window = leg[leg['retail_sales_year'].between(2010,2022)]['state'].tolist()

cdc_reg = cdc[cdc['state'].isin(in_window + never)].copy()
cdc_reg = cdc_reg.merge(leg[['state','retail_sales_year']], on='state', how='left')
cdc_reg['post'] = (
    cdc_reg['retail_sales_year'].notna() &
    (cdc_reg['year'] >= cdc_reg['retail_sales_year'])
).astype(float)

idx = cdc_reg.set_index(['state','year'])
fe  = PanelOLS(idx[outcome], idx[['post']],
               entity_effects=True, time_effects=True).fit(
               cov_type='clustered', cluster_entity=True)
b  = fe.params['post']
ci = fe.conf_int().loc['post']
print(f"TWFE ATT on {outcome}: {b:+.4f}  [{ci['lower']:+.4f}, {ci['upper']:+.4f}]")
print()
print("Negative and significant -> consistent with substitution hypothesis")
print("Zero or positive -> no evidence of substitution; may indicate complementarity")
print()
print("Note: opioid crisis was worsening throughout study period (confound).")
print("Event study pre-trends are especially important to check here.")